# NBA News Dataset (2020–2022)

Этот ноутбук формирует датасет новостей о командах NBA на основе архива GDELT.
Из raw-файлов извлекаются заголовки новостей, определяется команда, рассчитываются признаки тональности и формируются агрегированные признаки день–команда для последующего анализа и ML-моделей.

Изначально рассматривались различные новостные API, однако от них отказались из-за лимитов запросов и отсутствия полноценного исторического архива статей. Поэтому был выбран GDELT, предоставляющий открытый исторический корпус новостей.

## Установка

In [1]:
!pip -q install pandas pyarrow tqdm requests

## Импорты

In [2]:
import re
import io
import gc
import html
import zipfile
import requests
import pandas as pd

from pathlib import Path
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

## Папки

In [34]:
BASE_DIR = Path("/content/gdelt_nba")
OUT_DIR = BASE_DIR / "processed"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR:", OUT_DIR)

OUT_DIR: /content/gdelt_nba/processed


## Список команд NBA и алиасы

In [35]:
TEAM_ALIASES = {
    "ATL": ["atlanta hawks", "hawks"],
    "BOS": ["boston celtics", "celtics"],
    "BKN": ["brooklyn nets", "nets"],
    "CHA": ["charlotte hornets", "hornets"],
    "CHI": ["chicago bulls", "bulls"],
    "CLE": ["cleveland cavaliers", "cavaliers", "cavs"],
    "DAL": ["dallas mavericks", "mavericks", "mavs"],
    "DEN": ["denver nuggets", "nuggets"],
    "DET": ["detroit pistons", "pistons"],
    "GSW": ["golden state warriors", "warriors"],
    "HOU": ["houston rockets", "rockets"],
    "IND": ["indiana pacers", "pacers"],
    "LAC": ["los angeles clippers", "clippers"],
    "LAL": ["los angeles lakers", "lakers", "la lakers"],
    "MEM": ["memphis grizzlies", "grizzlies"],
    "MIA": ["miami heat", "heat"],
    "MIL": ["milwaukee bucks", "bucks"],
    "MIN": ["minnesota timberwolves", "timberwolves", "wolves"],
    "NOP": ["new orleans pelicans", "pelicans"],
    "NYK": ["new york knicks", "knicks"],
    "OKC": ["oklahoma city thunder", "thunder"],
    "ORL": ["orlando magic", "magic"],
    "PHI": ["philadelphia 76ers", "76ers", "sixers"],
    "PHX": ["phoenix suns", "suns"],
    "POR": ["portland trail blazers", "trail blazers", "blazers"],
    "SAC": ["sacramento kings", "kings"],
    "SAS": ["san antonio spurs", "spurs"],
    "TOR": ["toronto raptors", "raptors"],
    "UTA": ["utah jazz", "jazz"],
    "WAS": ["washington wizards", "wizards"],
}

print("Количество команд:", len(TEAM_ALIASES))
print(sorted(TEAM_ALIASES.keys()))

Количество команд: 30
['ATL', 'BKN', 'BOS', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']


## Словари тональности

In [39]:
positive_words = [

    # победы
    "win", "wins", "won", "victory", "victories",
    "beat", "beats", "beating",
    "defeat", "defeats", "defeated",
    "dominate", "dominates", "dominated", "dominating",

    # успех
    "success", "successful",
    "strong", "improve", "improved", "improves", "improving",
    "great", "excellent", "amazing",
    "best", "top", "elite",

    # камбэк
    "comeback", "return", "returns", "returned",
    "bounce back",

    # достижения
    "career-high", "record", "milestone",
    "breakthrough",

    # контракты
    "sign", "signed", "signs",
    "extension", "extended",

    # положительные новости
    "healthy", "cleared", "activated",
    "back", "available",

    # плейофф
    "playoff berth",
    "clinched",
    "advance", "advanced",
    "finals", "championship"
]

In [40]:
negative_words = [

    # поражения
    "lose", "loses", "lost", "loss",
    "defeat", "defeated",
    "fall", "falls", "fell",

    # травмы
    "injury", "injured", "injuries",
    "hurt", "hurts",
    "out", "ruled out",

    # проблемы
    "struggle", "struggles", "struggling",
    "weak", "collapse", "collapsed",

    # дисквалификации
    "suspended", "suspension",
    "ejected",

    # плохие результаты
    "worst", "poor", "bad",

    # обмены (иногда негативный контекст)
    "trade request",
    "waived",
    "released",

    # конфликты
    "controversy",
    "scandal",
    "investigation",

    # увольнения
    "fired",
    "dismissed",

    # поражения в серии
    "eliminated",
    "knocked out"
]

In [74]:
positive_patterns = [
    r"\bwin(?:s|ning)?\b",
    r"\bwon\b",
    r"\bvictor(?:y|ies)\b",
    r"\bbeat(?:s|en|ing)?\b",
    r"\bdominat(?:ed|es|ing)?\b",
    r"\bimprov(?:ed|es|ing)?\b",
    r"\bstrong\b",
    r"\bcomeback\b",
    r"\bgreat\b",
    r"\bexcellent\b",
    r"\bcareer[- ]high\b",
    r"\bmilestone\b",
    r"\brecord\b",
    r"\bhealthy\b",
    r"\bcleared\b",
    r"\bactivated\b",
    r"\breturn(?:s|ed|ing)?\b",
    r"\bclinch(?:ed|es|ing)?\b",
    r"\badvance(?:d|s|ing)?\b",
    r"\bchampionship\b",
]

negative_patterns = [
    r"\blose(?:s|ing)?\b",
    r"\blost\b",
    r"\bloss(?:es)?\b",
    r"\binjur(?:y|ies)\b",
    r"\binjured\b",
    r"\bhurt\b",
    r"\bruled out\b",
    r"\bdefeat(?:ed|s)?\b",
    r"\bstruggl(?:e|ed|es|ing)\b",
    r"\bweak\b",
    r"\bcollapse(?:d|s|ing)?\b",
    r"\bsuspend(?:ed|s|ing)?\b",
    r"\beject(?:ed|s|ing)?\b",
    r"\bworst\b",
    r"\bpoor\b",
    r"\bbad\b",
    r"\bwaiv(?:ed|es|ing)?\b",
    r"\breleased\b",
    r"\bfired\b",
    r"\beliminat(?:ed|es|ing)?\b",
]

def keyword_sentiment(text: str):
    t = normalize_text(text)

    pos = sum(len(re.findall(p, t)) for p in positive_patterns)
    neg = sum(len(re.findall(p, t)) for p in negative_patterns)
    score = pos - neg

    if score > 0:
        label = "positive"
    elif score < 0:
        label = "negative"
    else:
        label = "neutral"

    return label, score, pos, neg

## Колонки GKG

In [38]:
GKG_COLUMNS = [
    "GKGRECORDID",
    "DATE",
    "SourceCollectionIdentifier",
    "SourceCommonName",
    "DocumentIdentifier",
    "Counts",
    "V2Counts",
    "Themes",
    "V2Themes",
    "Locations",
    "V2Locations",
    "Persons",
    "V2Persons",
    "Organizations",
    "V2Organizations",
    "V2Tone",
    "Dates",
    "GCAM",
    "SharingImage",
    "RelatedImages",
    "SocialImageEmbeds",
    "SocialVideoEmbeds",
    "Quotations",
    "AllNames",
    "Amounts",
    "TranslationInfo",
    "Extras",
]

## Общие regex-паттерны

In [72]:
NBA_CONTEXT_WORDS = [
    "nba", "basketball", "playoff", "playoffs", "season", "game", "games",
    "coach", "roster", "trade", "draft", "finals", "western conference",
    "eastern conference", "all-star", "matchup", "preseason", "postseason",
    "tipoff", "backcourt", "frontcourt", "starting lineup"
]

SPORTS_SOURCE_REGEX = re.compile(
    r"espn|nba|sports|bleacher|athletic|yahoo|foxsports|cbssports",
    flags=re.IGNORECASE
)

NBA_CONTEXT_REGEX = re.compile(
    r"|".join(re.escape(w) for w in NBA_CONTEXT_WORDS),
    flags=re.IGNORECASE
)

TEAM_REGEX = {}
for team, aliases in TEAM_ALIASES.items():
    alias_patterns = []
    for alias in aliases:
        alias_patterns.append(rf"(^|[^a-z]){re.escape(alias)}([^a-z]|$)")
    TEAM_REGEX[team] = re.compile("|".join(alias_patterns), flags=re.IGNORECASE)

## Базовые функции

In [43]:
def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = html.unescape(str(text)).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [44]:
def extract_page_title(extras: str) -> str:
    if not isinstance(extras, str) or "<PAGE_TITLE>" not in extras:
        return ""
    m = re.search(r"<PAGE_TITLE>(.*?)</PAGE_TITLE>", extras, flags=re.DOTALL)
    if not m:
        return ""
    return html.unescape(m.group(1)).strip()

In [45]:
def parse_gdelt_tone(v2tone: str):
    if pd.isna(v2tone):
        return None
    first = str(v2tone).split(",")[0].strip()
    try:
        return float(first)
    except:
        return None

In [73]:
def detect_teams(title: str):
    t = normalize_text(title)
    matched = []

    for team, pattern in TEAM_REGEX.items():
        if pattern.search(t):
            matched.append(team)

    return matched

In [75]:
def is_nba_relevant(title: str, source_name: str = "") -> bool:
    title_norm = normalize_text(title)
    source_name = "" if source_name is None else str(source_name)

    has_team = len(detect_teams(title_norm)) > 0
    has_context = bool(NBA_CONTEXT_REGEX.search(title_norm))
    sports_source = bool(SPORTS_SOURCE_REGEX.search(source_name))

    return has_team and (has_context or sports_source)

## Параметры сбора

In [48]:
BASE_GKG_URL = "http://data.gdeltproject.org/gdeltv2"

START_DT = datetime(2020, 1, 1, 0, 0, 0)
END_DT   = datetime(2022, 12, 31, 18, 0, 0)

FREQ_HOURS = 6
MAX_WORKERS = 4
REQUEST_TIMEOUT = 60

## Генерация списка URL

In [49]:
def generate_gkg_urls(start_dt: datetime, end_dt: datetime, freq_hours: int = 6):
    rows = []
    current = start_dt

    while current <= end_dt:
        ts = current.strftime("%Y%m%d%H%M%S")
        url = f"{BASE_GKG_URL}/{ts}.gkg.csv.zip"
        rows.append({"dt": current, "url": url})
        current += timedelta(hours=freq_hours)

    return pd.DataFrame(rows)

In [50]:
files_df = generate_gkg_urls(START_DT, END_DT, FREQ_HOURS)

print(files_df.shape)
files_df.head()

(4384, 2)


,dt,url
0,2020-01-01 00:00:00,http://data.gdeltproject.org/gdeltv2/202001010...
1,2020-01-01 06:00:00,http://data.gdeltproject.org/gdeltv2/202001010...
2,2020-01-01 12:00:00,http://data.gdeltproject.org/gdeltv2/202001011...
3,2020-01-01 18:00:00,http://data.gdeltproject.org/gdeltv2/202001011...
4,2020-01-02 00:00:00,http://data.gdeltproject.org/gdeltv2/202001020...


## Разбивка по месяцам

In [51]:
files_df["year_month"] = files_df["dt"].dt.strftime("%Y-%m")
month_groups = dict(tuple(files_df.groupby("year_month")))

print("Месяцев:", len(month_groups))
print(list(month_groups.keys())[:5])

Месяцев: 36
['2020-01', '2020-02', '2020-03', '2020-04', '2020-05']


## Сессия requests

In [52]:
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 Colab GDELT NBA collector"
})

## Обработка одного raw GKG файла

In [76]:
def process_gkg_file(url: str):
    try:
        resp = session.get(url, timeout=REQUEST_TIMEOUT)
        if resp.status_code != 200:
            return pd.DataFrame()

        zf = zipfile.ZipFile(io.BytesIO(resp.content))
        inner_name = zf.namelist()[0]

        with zf.open(inner_name) as f:
            df = pd.read_csv(
                f,
                sep="\t",
                header=None,
                names=GKG_COLUMNS,
                usecols=[1, 3, 4, 15, 26],
                dtype=str,
                keep_default_na=False,
                on_bad_lines="skip",
            )

        df.columns = ["DATE", "SourceCommonName", "DocumentIdentifier", "V2Tone", "Extras"]

        df["title"] = df["Extras"].map(extract_page_title)

        df = df[df["title"].str.len() > 5].copy()
        if df.empty:
            return pd.DataFrame()

        df["is_nba_relevant"] = [
            is_nba_relevant(title, source)
            for title, source in zip(df["title"], df["SourceCommonName"])
        ]
        df = df[df["is_nba_relevant"]].copy()
        if df.empty:
            return pd.DataFrame()

        df["teams"] = df["title"].map(detect_teams)
        df = df[df["teams"].map(len) > 0].copy()
        if df.empty:
            return pd.DataFrame()

        df = df.explode("teams").rename(columns={"teams": "team"})

        df["news_date"] = pd.to_datetime(
            df["DATE"], format="%Y%m%d%H%M%S", errors="coerce"
        ).dt.date

        df["gdelt_tone"] = df["V2Tone"].map(parse_gdelt_tone)

        sent = df["title"].map(keyword_sentiment)
        df["sentiment_label_kw"] = sent.map(lambda x: x[0])
        df["sentiment_score_kw"] = sent.map(lambda x: x[1])
        df["positive_hits"] = sent.map(lambda x: x[2])
        df["negative_hits"] = sent.map(lambda x: x[3])

        df["has_injury_kw"] = df["title"].str.contains(
            r"\binjur(?:y|ies|ed)\b|\bhurt\b|\bruled out\b",
            case=False,
            regex=True,
            na=False
        ).astype(int)

        df["has_win_kw"] = df["title"].str.contains(
            r"\bwin(?:s|ning)?\b|\bwon\b|\bvictor(?:y|ies)\b|\bbeat(?:s|en|ing)?\b",
            case=False,
            regex=True,
            na=False
        ).astype(int)

        df["has_lose_kw"] = df["title"].str.contains(
            r"\blose(?:s|ing)?\b|\blost\b|\bloss(?:es)?\b",
            case=False,
            regex=True,
            na=False
        ).astype(int)

        df["has_trade_kw"] = df["title"].str.contains(
            r"\btrade\b|\btraded\b|\bdeadline\b",
            case=False,
            regex=True,
            na=False
        ).astype(int)

        df["has_playoff_kw"] = df["title"].str.contains(
            r"\bplayoff(?:s)?\b|\bfinals\b|\bpostseason\b|\bclinch(?:ed|es|ing)?\b",
            case=False,
            regex=True,
            na=False
        ).astype(int)

        df["title_len"] = df["title"].str.len()
        df["title_word_count"] = df["title"].str.split().str.len()

        df = df.rename(columns={
            "SourceCommonName": "source_name",
            "DocumentIdentifier": "url"
        })

        return df[[
            "news_date",
            "team",
            "title",
            "url",
            "source_name",
            "gdelt_tone",
            "sentiment_label_kw",
            "sentiment_score_kw",
            "positive_hits",
            "negative_hits",
            "has_injury_kw",
            "has_win_kw",
            "has_lose_kw",
            "has_trade_kw",
            "has_playoff_kw",
            "title_len",
            "title_word_count",
        ]].copy()

    except Exception:
        return pd.DataFrame()

## Обработка одного месяца

In [54]:
def process_month(month_key: str, month_df: pd.DataFrame, out_dir: Path, max_workers: int = 4):
    month_path = out_dir / f"nba_news_{month_key}.parquet"

    if month_path.exists():
        print(f"skip {month_key}: already exists")
        return month_path

    urls = month_df["url"].tolist()
    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(process_gkg_file, url) for url in urls]

        for fut in tqdm(as_completed(futures), total=len(futures), desc=month_key):
            df = fut.result()
            if not df.empty:
                results.append(df)

    if results:
        month_news = pd.concat(results, ignore_index=True)
        month_news = month_news.drop_duplicates(subset=["team", "title", "url"]).reset_index(drop=True)
    else:
        month_news = pd.DataFrame(columns=[
            "news_date", "team", "title", "url", "source_name",
            "gdelt_tone", "sentiment_label_kw", "sentiment_score_kw",
            "positive_hits", "negative_hits"
        ])

    month_news.to_parquet(month_path, index=False)
    print(month_key, "saved:", month_news.shape)

    gc.collect()
    return month_path

## Функции анализа и очистки

In [55]:
def load_month(month: str, out_dir: Path) -> pd.DataFrame:
    path = out_dir / f"nba_news_{month}.parquet"
    df = pd.read_parquet(path)
    print(f"{month} loaded:", df.shape)
    return df

In [56]:
def show_month_stats(df: pd.DataFrame):
    print("shape:", df.shape)
    print("\nunique teams:", df["team"].nunique())
    print(sorted(df["team"].unique())[:30])

    print("\nteam counts:")
    print(df["team"].value_counts().head(15))

    print("\nsource counts:")
    print(df["source_name"].fillna("NA").value_counts().head(15))

    print("\nsentiment counts:")
    print(df["sentiment_label_kw"].value_counts(dropna=False))

    print("\ndate range:")
    print(df["news_date"].min(), "->", df["news_date"].max())

In [77]:
def apply_light_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = df.drop_duplicates(subset=["team", "title", "url"]).reset_index(drop=True)
    df = df[df["title"].str.len() > 5].copy()

    return df

In [78]:
def build_nlp_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "title_len" not in df.columns:
        df["title_len"] = df["title"].str.len()

    if "title_word_count" not in df.columns:
        df["title_word_count"] = df["title"].str.split().str.len()

    if "has_injury_kw" not in df.columns:
        df["has_injury_kw"] = df["title"].str.contains(
            r"\binjur(?:y|ies|ed)\b|\bhurt\b|\bruled out\b",
            case=False,
            regex=True,
            na=False
        ).astype(int)

    if "has_win_kw" not in df.columns:
        df["has_win_kw"] = df["title"].str.contains(
            r"\bwin(?:s|ning)?\b|\bwon\b",
            case=False,
            regex=True,
            na=False
        ).astype(int)

    if "has_lose_kw" not in df.columns:
        df["has_lose_kw"] = df["title"].str.contains(
            r"\blose(?:s|ing)?\b|\blost\b",
            case=False,
            regex=True,
            na=False
        ).astype(int)

    return df

In [59]:
def build_daily_features(df: pd.DataFrame) -> pd.DataFrame:
    daily = (
        df.groupby(["news_date", "team"], as_index=False)
        .agg(
            news_count=("title", "count"),
            avg_gdelt_tone=("gdelt_tone", "mean"),
            sum_positive_hits=("positive_hits", "sum"),
            sum_negative_hits=("negative_hits", "sum"),
            avg_sentiment_score_kw=("sentiment_score_kw", "mean"),
            injury_news_count=("has_injury_kw", "sum"),
            win_kw_count=("has_win_kw", "sum"),
            lose_kw_count=("has_lose_kw", "sum"),
        )
    )
    return daily

## Тест на одном месяце

In [79]:
test_month = "2020-01"
process_month(test_month, month_groups[test_month], OUT_DIR, max_workers=MAX_WORKERS)

2020-01:   0%|          | 0/124 [00:00<?, ?it/s]

2020-01 saved: (407, 17)


PosixPath('/content/gdelt_nba/processed/nba_news_2020-01.parquet')

## Чистим январь и строим признаки

In [80]:
jan_df = load_month("2020-01", OUT_DIR)
show_month_stats(jan_df)

2020-01 loaded: (407, 17)
shape: (407, 17)

unique teams: 29
['ATL', 'BKN', 'BOS', 'CHA', 'CHI', 'CLE', 'DAL', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']

team counts:
team
LAL    70
LAC    34
PHI    23
GSW    21
MIL    21
SAC    19
BOS    17
NYK    17
MIN    17
NOP    14
OKC    13
BKN    13
MIA    12
DAL    10
POR    10
Name: count, dtype: int64

source counts:
source_name
iheart.com             45
bleachernation.com     15
yahoo.com              14
bleacherreport.com     14
ibtimes.com            13
clutchpoints.com       13
espn.com               13
kesn1033.com           10
lakersnation.com        9
freep.com               8
nba.com                 7
oklahomacitysun.com     6
thefanfm.com            6
ktop1490.com            6
usatoday.com            5
Name: count, dtype: int64

sentiment counts:
sentiment_label_kw
neutral     336
negative     36
positive     35
Name: coun

In [81]:
jan_df.head(20)

,news_date,team,title,url,source_name,gdelt_tone,sentiment_label_kw,sentiment_score_kw,positive_hits,negative_hits,has_injury_kw,has_win_kw,has_lose_kw,has_trade_kw,has_playoff_kw,title_len,title_word_count
0,2020-01-01,MIL,NBA-best Bucks take on struggling Timberwolves,http://www.kesn1033.com/news/nba-best-bucks-ta...,kesn1033.com,1.898734,negative,-1,0,1,0,0,0,0,0,46,6
1,2020-01-01,MIN,NBA-best Bucks take on struggling Timberwolves,http://www.kesn1033.com/news/nba-best-bucks-ta...,kesn1033.com,1.898734,negative,-1,0,1,0,0,0,0,0,46,6
2,2020-01-01,GSW,"Warriors look like a new team in thrilling, un...",http://www.knbr.com/2019/12/31/warriors-look-l...,knbr.com,-1.325967,neutral,0,0,0,0,0,0,0,0,70,11
3,2020-01-01,SAS,"Warriors look like a new team in thrilling, un...",http://www.knbr.com/2019/12/31/warriors-look-l...,knbr.com,-1.325967,neutral,0,0,0,0,0,0,0,0,70,11
4,2020-01-01,GSW,Road warriors: Seahawks hoping away success co...,https://www.espn.com/blog/nfcwest/post/_/id/13...,espn.com,0.902062,neutral,0,0,0,0,0,0,0,1,67,9
5,2020-01-01,LAL,Suns At Lakers 01/01/20: Odds And NBA Betting ...,https://www.lakersnation.com/suns-at-lakers-01...,lakersnation.com,0.223714,neutral,0,0,0,0,0,0,0,0,52,9
6,2020-01-01,PHX,Suns At Lakers 01/01/20: Odds And NBA Betting ...,https://www.lakersnation.com/suns-at-lakers-01...,lakersnation.com,0.223714,neutral,0,0,0,0,0,0,0,0,52,9
7,2020-01-02,MIL,Shorthanded Wolves Give Bucks All They've Got ...,https://www.nba.com/timberwolves/news/shorthan...,nba.com,-0.503778,neutral,0,0,0,0,0,0,0,0,70,10
8,2020-01-02,MIN,Shorthanded Wolves Give Bucks All They've Got ...,https://www.nba.com/timberwolves/news/shorthan...,nba.com,-0.503778,neutral,0,0,0,0,0,0,0,0,70,10
9,2020-01-02,NYK,New York Knicks spoil Carmelo Anthony's return...,https://in.nba.com/news/new-york-knicks-spoil-...,nba.com,0.566038,positive,2,2,0,0,0,0,0,0,134,19


In [82]:
jan_df.sample(20, random_state=42)[["news_date", "team", "title", "source_name"]]

,news_date,team,title,source_name
70,2020-01-08,NYK,NBA: How Michael Jordan Almost Signed With New...,ibtimes.com
218,2020-01-22,NYK,LaVar Ball's plan to save the Knicks: Draft La...,nypost.com
258,2020-01-26,OKC,NBA notebook: Thunder C Noel out after cheek s...,oklahomacitysun.com
33,2020-01-05,WAS,NBA notebook: Contact with ref costs Wizards' ...,oann.com
42,2020-01-05,MEM,Preview: Suns and Grizzlies battle for early p...,brightsideofthesun.com
77,2020-01-09,CHA,NBA roundup: Raptors drop Hornets in overtime,pressherald.com
137,2020-01-16,NOP,Zion Williamson Set To Make NBA Debut For New ...,kbur.com
334,2020-01-28,LAL,"NBA Postpones Tuesday's Lakers, Clippers Game ...",iheart.com
245,2020-01-25,MIL,"NBA roundup: Giannis, Bucks light up Paris",middleeaststar.com
266,2020-01-26,OKC,"Paul, Thunder defeat Wolves for third time thi...",kesn1033.com


In [83]:
jan_clean = apply_nba_context_filter(jan_df)
jan_nlp = build_nlp_features(jan_clean)
jan_daily = build_daily_features(jan_nlp)

print("raw:", jan_df.shape)
print("clean:", jan_clean.shape)
print("nlp:", jan_nlp.shape)
print("daily:", jan_daily.shape)

raw: (407, 17)
clean: (407, 19)
nlp: (407, 19)
daily: (247, 10)


In [84]:
jan_nlp.head()

,news_date,team,title,url,source_name,gdelt_tone,sentiment_label_kw,sentiment_score_kw,positive_hits,negative_hits,has_injury_kw,has_win_kw,has_lose_kw,has_trade_kw,has_playoff_kw,title_len,title_word_count,nba_context_hit,source_is_sports
0,2020-01-01,MIL,NBA-best Bucks take on struggling Timberwolves,http://www.kesn1033.com/news/nba-best-bucks-ta...,kesn1033.com,1.898734,negative,-1,0,1,0,0,0,0,0,46,6,1,0
1,2020-01-01,MIN,NBA-best Bucks take on struggling Timberwolves,http://www.kesn1033.com/news/nba-best-bucks-ta...,kesn1033.com,1.898734,negative,-1,0,1,0,0,0,0,0,46,6,1,0
2,2020-01-01,GSW,"Warriors look like a new team in thrilling, un...",http://www.knbr.com/2019/12/31/warriors-look-l...,knbr.com,-1.325967,neutral,0,0,0,0,0,0,0,0,70,11,1,0
3,2020-01-01,SAS,"Warriors look like a new team in thrilling, un...",http://www.knbr.com/2019/12/31/warriors-look-l...,knbr.com,-1.325967,neutral,0,0,0,0,0,0,0,0,70,11,1,0
4,2020-01-01,GSW,Road warriors: Seahawks hoping away success co...,https://www.espn.com/blog/nfcwest/post/_/id/13...,espn.com,0.902062,neutral,0,0,0,0,0,0,0,1,67,9,1,1


In [85]:
jan_daily.head()

,news_date,team,news_count,avg_gdelt_tone,sum_positive_hits,sum_negative_hits,avg_sentiment_score_kw,injury_news_count,win_kw_count,lose_kw_count
0,2020-01-01,GSW,2,-0.211952,0,0,0.0,0,0,0
1,2020-01-01,LAL,1,0.223714,0,0,0.0,0,0,0
2,2020-01-01,MIL,1,1.898734,0,1,-1.0,0,0,0
3,2020-01-01,MIN,1,1.898734,0,1,-1.0,0,0,0
4,2020-01-01,PHX,1,0.223714,0,0,0.0,0,0,0


In [86]:
jan_clean["team"].value_counts().head(20)

,count
team,
LAL,70
LAC,34
PHI,23
GSW,21
MIL,21
SAC,19
BOS,17
NYK,17
MIN,17


In [87]:
jan_clean["source_name"].value_counts().head(20)

,count
source_name,
iheart.com,45
bleachernation.com,15
yahoo.com,14
bleacherreport.com,14
ibtimes.com,13
clutchpoints.com,13
espn.com,13
kesn1033.com,10
lakersnation.com,9


In [88]:
jan_clean.sample(20)[["team","title"]]

,team,title
89,BOS,"With Joel Embiid out, Ben Simmons, Al Horford ..."
153,MIN,T-Wolves' Towns returns after 15-game absence
331,LAC,NBA Postpones Tuesday's Lakers-Clippers Game A...
385,SAS,"Spurs outplay the Jazz, snap three game skid"
247,DAL,Warriors news: Golden State trades Willie Caul...
18,MIA,"BBL, Big Bash League 2020, Hobart Hurricanes v..."
175,LAL,Celtics aim to end three-game skid vs. red-hot...
112,LAL,An appreciation of the efficient anomaly that ...
62,BOS,"""Possibility"" Sixers' center Joel Embiid could..."
106,LAL,NBA Betting - Lakers +115


## Запуск на весь период

In [89]:
for month_key, month_df in month_groups.items():
    process_month(month_key, month_df, OUT_DIR, max_workers=MAX_WORKERS)

skip 2020-01: already exists


2020-02:   0%|          | 0/116 [00:00<?, ?it/s]

2020-02 saved: (406, 17)


2020-03:   0%|          | 0/124 [00:00<?, ?it/s]

2020-03 saved: (240, 17)


...

2022-10 saved: (230, 17)


2022-11:   0%|          | 0/120 [00:00<?, ?it/s]

2022-11 saved: (264, 17)


2022-12:   0%|          | 0/124 [00:00<?, ?it/s]

2022-12 saved: (313, 17)


In [90]:
def load_all_months(out_dir: Path) -> pd.DataFrame:
    parts = sorted(out_dir.glob("nba_news_*.parquet"))
    print("parts:", len(parts))

    if not parts:
        return pd.DataFrame()

    all_df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    all_df = all_df.drop_duplicates(subset=["team", "title", "url"]).reset_index(drop=True)
    return all_df

In [91]:
all_df = load_all_months(OUT_DIR)
print(all_df.shape)
all_df.head()

parts: 36
(9927, 17)


,news_date,team,title,url,source_name,gdelt_tone,sentiment_label_kw,sentiment_score_kw,positive_hits,negative_hits,has_injury_kw,has_win_kw,has_lose_kw,has_trade_kw,has_playoff_kw,title_len,title_word_count
0,2020-01-01,MIL,NBA-best Bucks take on struggling Timberwolves,http://www.kesn1033.com/news/nba-best-bucks-ta...,kesn1033.com,1.898734,negative,-1,0,1,0,0,0,0,0,46,6
1,2020-01-01,MIN,NBA-best Bucks take on struggling Timberwolves,http://www.kesn1033.com/news/nba-best-bucks-ta...,kesn1033.com,1.898734,negative,-1,0,1,0,0,0,0,0,46,6
2,2020-01-01,GSW,"Warriors look like a new team in thrilling, un...",http://www.knbr.com/2019/12/31/warriors-look-l...,knbr.com,-1.325967,neutral,0,0,0,0,0,0,0,0,70,11
3,2020-01-01,SAS,"Warriors look like a new team in thrilling, un...",http://www.knbr.com/2019/12/31/warriors-look-l...,knbr.com,-1.325967,neutral,0,0,0,0,0,0,0,0,70,11
4,2020-01-01,GSW,Road warriors: Seahawks hoping away success co...,https://www.espn.com/blog/nfcwest/post/_/id/13...,espn.com,0.902062,neutral,0,0,0,0,0,0,0,1,67,9


## Финальный dataset

In [92]:
clean_df = apply_nba_context_filter(all_df)
nlp_df = build_nlp_features(clean_df)
daily_news_features = build_daily_features(nlp_df)

print("all raw:", all_df.shape)
print("all clean:", clean_df.shape)
print("all nlp:", nlp_df.shape)
print("all daily:", daily_news_features.shape)

all raw: (9927, 17)
all clean: (9881, 19)
all nlp: (9881, 19)
all daily: (6155, 10)
